# 00 - Train and Compare ACGAN vs DCGAN

Generated for Colab. The repo URL is set to `https://github.com/tlinhevg05/contrastive-synthesis-medcls_CVProject.git`.


## Before running
These notebooks are designed for **Colab**.

The real processed data is expected inside the cloned repo:

```text
/content/contrastive-synthesis-medcls_CVProject/data/processed/
├── labelled_4232/
│   ├── COVID/images/
│   ├── Lung_Opacity/images/
│   ├── Viral_Pneumonia/images/
│   └── Normal/images/
└── unlabelled_16934/images/
```

Notebook `00_train_compare_gans_acgan_dcgan.ipynb` saves generated synthetic images to **Google Drive** so they persist after Colab disconnects:

```text
/content/drive/MyDrive/medcls_cvproject/data/processed/synthetic_dcgan/
/content/drive/MyDrive/medcls_cvproject/data/processed/synthetic_acgan/
```

The synthetic classification notebooks (`05`, `06`, `11`, `12`) load `synthetic_dcgan` from Google Drive, so run notebook `00` first.


In [ ]:

# =========================
# 1. Colab / Drive / Repo setup
# =========================
import os, sys, json, math, random, time, copy, subprocess
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Drive mount skipped:', e)

REPO_URL = "https://github.com/tlinhevg05/contrastive-synthesis-medcls_CVProject.git"
REPO_DIR = Path('/content/contrastive-synthesis-medcls_CVProject')
DRIVE_ROOT = Path('/content/drive/MyDrive/medcls_cvproject')
REPO_DATA_ROOT = REPO_DIR / 'data' / 'processed'
DRIVE_DATA_ROOT = DRIVE_ROOT / 'data' / 'processed'
LABELLED_DATA = REPO_DATA_ROOT / 'labelled_4232'
OUTPUT_DIR = DRIVE_ROOT / 'outputs' / 'gan_compare_acgan_dcgan'
SYNTHETIC_DCGAN_OUT = DRIVE_DATA_ROOT / 'synthetic_dcgan'
SYNTHETIC_ACGAN_OUT = DRIVE_DATA_ROOT / 'synthetic_acgan'
CLASSES = ['COVID', 'Lung_Opacity', 'Viral_Pneumonia', 'Normal']
SEED = 42

# Keep True for report-style training. Set False for a quick smoke test.
FULL_RUN = True
FORCE_RETRAIN = False
COMPUTE_IS_FID = True

# Report-like GAN settings.
IMG_SIZE = 64
NZ = 100
BATCH_SIZE = 64 if FULL_RUN else 16
EPOCHS = 100 if FULL_RUN else 1
LR = 2e-5
D_UPDATE_EVERY = 3
N_SYNTH_PER_CLASS = 1200 if FULL_RUN else 24
METRIC_MAX_IMAGES = 2000 if FULL_RUN else 64

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_DATA_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SYNTHETIC_DCGAN_OUT.mkdir(parents=True, exist_ok=True)
SYNTHETIC_ACGAN_OUT.mkdir(parents=True, exist_ok=True)

print('Repo labelled data:', LABELLED_DATA)
print('Drive synthetic DCGAN output:', SYNTHETIC_DCGAN_OUT)
print('Drive synthetic ACGAN output:', SYNTHETIC_ACGAN_OUT)
print('Metrics/checkpoints output:', OUTPUT_DIR)

if not REPO_DIR.exists():
    result = subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], text=True, capture_output=True)
    print(result.stdout)
    if result.returncode != 0:
        print('WARNING: repo clone failed. Check repo URL / visibility. Error:')
        print(result.stderr)
else:
    print('Repo already exists:', REPO_DIR)

if REPO_DIR.exists():
    os.chdir(REPO_DIR)
    print('Working directory:', Path.cwd())

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'])
if (REPO_DIR / 'requirements.txt').exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'requirements.txt')])
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-learn', 'scipy', 'pandas', 'tqdm', 'seaborn', 'matplotlib'])


In [ ]:

# =========================
# 2. Imports, datasets, transforms
# =========================
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, utils as vutils
from torchvision.models import inception_v3, Inception_V3_Weights
from scipy.linalg import sqrtm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU, running on CPU')
print('Device:', DEVICE)

IMG_EXTS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True
set_seed(SEED)

def list_images(root):
    root = Path(root)
    if not root.exists():
        return []
    return [p for p in root.rglob('*') if p.suffix.lower() in IMG_EXTS and p.is_file()]

def class_image_dir(root, cls):
    croot = Path(root) / cls
    return croot / 'images' if (croot / 'images').exists() else croot

class ClassImageDataset(Dataset):
    def __init__(self, root, classes, transform=None, selected_class=None, max_per_class=None):
        self.samples = []
        self.classes = classes
        self.transform = transform
        for label, cls in enumerate(classes):
            if selected_class is not None and cls != selected_class:
                continue
            files = sorted(list_images(class_image_dir(root, cls)))
            if max_per_class is not None:
                files = files[:max_per_class]
            self.samples.extend([(p, label) for p in files])
        if not self.samples:
            raise ValueError(f'No images found in {root}')
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

gan_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

for cls in CLASSES:
    print(cls, len(list_images(class_image_dir(LABELLED_DATA, cls))))
assert LABELLED_DATA.exists(), f'Missing labelled data: {LABELLED_DATA}'


In [ ]:

# =========================
# 3. DCGAN and ACGAN models
# =========================
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1 or classname.find('Linear') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

class DCGANGenerator(nn.Module):
    def __init__(self, nz=100, ngf=64, nc=3):
        super().__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(nz, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8), nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4), nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2), nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf), nn.ReLU(True),
            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),
            nn.Tanh(),
        )
    def forward(self, z):
        return self.main(z)

class DCGANDiscriminator(nn.Module):
    def __init__(self, nc=3, ndf=64):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True), nn.Dropout2d(0.2),
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2), nn.LeakyReLU(0.2, inplace=True), nn.Dropout2d(0.2),
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4), nn.LeakyReLU(0.2, inplace=True), nn.Dropout2d(0.2),
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8), nn.LeakyReLU(0.2, inplace=True), nn.Dropout2d(0.2),
            nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False),
        )
    def forward(self, x):
        return self.main(x).view(-1)

class ACGANGenerator(nn.Module):
    def __init__(self, nz=100, num_classes=4, emb_dim=50, ngf=64, nc=3):
        super().__init__()
        self.embed = nn.Embedding(num_classes, emb_dim)
        self.net = DCGANGenerator(nz + emb_dim, ngf, nc)
    def forward(self, z, labels):
        emb = self.embed(labels).view(labels.size(0), -1, 1, 1)
        x = torch.cat([z, emb], dim=1)
        return self.net(x)

class ACGANDiscriminator(nn.Module):
    def __init__(self, num_classes=4, nc=3, ndf=64):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True), nn.Dropout2d(0.2),
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2), nn.LeakyReLU(0.2, inplace=True), nn.Dropout2d(0.2),
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4), nn.LeakyReLU(0.2, inplace=True), nn.Dropout2d(0.2),
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8), nn.LeakyReLU(0.2, inplace=True), nn.Dropout2d(0.2),
        )
        self.source = nn.Linear(ndf * 8 * 4 * 4, 1)
        self.classifier = nn.Linear(ndf * 8 * 4 * 4, num_classes)
    def forward(self, x):
        h = self.features(x).view(x.size(0), -1)
        return self.source(h).view(-1), self.classifier(h)


In [ ]:

# =========================
# 4. Train DCGAN: one generator per class
# =========================
def denorm(x):
    return (x * 0.5 + 0.5).clamp(0, 1)

def save_sample_grid(images, path, nrow=8):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    vutils.save_image(denorm(images.detach().cpu()), str(path), nrow=nrow)

def train_dcgan_for_class(cls):
    ckpt_path = OUTPUT_DIR / 'checkpoints' / f'dcgan_{cls}.pt'
    if ckpt_path.exists() and not FORCE_RETRAIN:
        print('Using existing DCGAN checkpoint:', ckpt_path)
        return ckpt_path

    dataset = ClassImageDataset(LABELLED_DATA, CLASSES, transform=gan_tf, selected_class=cls, max_per_class=None if FULL_RUN else 64)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, drop_last=True, pin_memory=True)
    G = DCGANGenerator(NZ).to(DEVICE).apply(weights_init)
    D = DCGANDiscriminator().to(DEVICE).apply(weights_init)
    optG = torch.optim.Adam(G.parameters(), lr=LR, betas=(0.5, 0.999))
    optD = torch.optim.Adam(D.parameters(), lr=LR, betas=(0.5, 0.999))
    criterion = nn.BCEWithLogitsLoss()

    history = []
    global_step = 0
    fixed_noise = torch.randn(32, NZ, 1, 1, device=DEVICE)

    for epoch in range(EPOCHS):
        g_losses, d_losses = [], []
        for real, _ in tqdm(loader, desc=f'DCGAN {cls} epoch {epoch+1}/{EPOCHS}'):
            real = real.to(DEVICE)
            b = real.size(0)
            valid = torch.ones(b, device=DEVICE)
            fake_lab = torch.zeros(b, device=DEVICE)

            if global_step % D_UPDATE_EVERY == 0:
                z = torch.randn(b, NZ, 1, 1, device=DEVICE)
                fake = G(z).detach()
                d_real = criterion(D(real), valid)
                d_fake = criterion(D(fake), fake_lab)
                d_loss = d_real + d_fake
                optD.zero_grad(set_to_none=True)
                d_loss.backward()
                optD.step()
                d_losses.append(d_loss.item())

            z = torch.randn(b, NZ, 1, 1, device=DEVICE)
            fake = G(z)
            g_loss = criterion(D(fake), valid)
            optG.zero_grad(set_to_none=True)
            g_loss.backward()
            optG.step()
            g_losses.append(g_loss.item())
            global_step += 1

        row = {'epoch': epoch+1, 'class': cls, 'g_loss': float(np.mean(g_losses)), 'd_loss': float(np.mean(d_losses)) if d_losses else None}
        history.append(row)
        print(row)
        if (epoch + 1) % max(1, EPOCHS // 5) == 0 or epoch == 0:
            save_sample_grid(G(fixed_noise), OUTPUT_DIR / 'samples' / f'dcgan_{cls}_epoch_{epoch+1}.png')

    ckpt_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({'G': G.state_dict(), 'D': D.state_dict(), 'class': cls, 'classes': CLASSES}, ckpt_path)
    pd.DataFrame(history).to_csv(OUTPUT_DIR / f'dcgan_{cls}_history.csv', index=False)
    print('Saved:', ckpt_path)
    return ckpt_path

DCGAN_CKPTS = {}
for cls in CLASSES:
    DCGAN_CKPTS[cls] = train_dcgan_for_class(cls)


In [ ]:

# =========================
# 5. Train ACGAN: one conditional generator for all classes
# =========================
def train_acgan():
    ckpt_path = OUTPUT_DIR / 'checkpoints' / 'acgan.pt'
    if ckpt_path.exists() and not FORCE_RETRAIN:
        print('Using existing ACGAN checkpoint:', ckpt_path)
        return ckpt_path

    dataset = ClassImageDataset(LABELLED_DATA, CLASSES, transform=gan_tf, max_per_class=None if FULL_RUN else 64)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, drop_last=True, pin_memory=True)
    G = ACGANGenerator(NZ, len(CLASSES)).to(DEVICE).apply(weights_init)
    D = ACGANDiscriminator(len(CLASSES)).to(DEVICE).apply(weights_init)
    optG = torch.optim.Adam(G.parameters(), lr=LR, betas=(0.5, 0.999))
    optD = torch.optim.Adam(D.parameters(), lr=LR, betas=(0.5, 0.999))
    bce = nn.BCEWithLogitsLoss()
    ce = nn.CrossEntropyLoss()
    history = []
    fixed_noise = torch.randn(32, NZ, 1, 1, device=DEVICE)
    fixed_labels = torch.tensor([i % len(CLASSES) for i in range(32)], device=DEVICE)

    for epoch in range(EPOCHS):
        g_losses, d_losses = [], []
        for real, labels in tqdm(loader, desc=f'ACGAN epoch {epoch+1}/{EPOCHS}'):
            real, labels = real.to(DEVICE), labels.to(DEVICE)
            b = real.size(0)
            valid = torch.ones(b, device=DEVICE)
            fake_lab = torch.zeros(b, device=DEVICE)
            sampled_labels = torch.randint(0, len(CLASSES), (b,), device=DEVICE)
            z = torch.randn(b, NZ, 1, 1, device=DEVICE)
            fake = G(z, sampled_labels).detach()

            real_src, real_cls = D(real)
            fake_src, fake_cls = D(fake)
            d_loss = bce(real_src, valid) + bce(fake_src, fake_lab) + ce(real_cls, labels) + ce(fake_cls, sampled_labels)
            optD.zero_grad(set_to_none=True)
            d_loss.backward()
            optD.step()

            z = torch.randn(b, NZ, 1, 1, device=DEVICE)
            sampled_labels = torch.randint(0, len(CLASSES), (b,), device=DEVICE)
            fake = G(z, sampled_labels)
            src, cls_logits = D(fake)
            g_loss = bce(src, valid) + ce(cls_logits, sampled_labels)
            optG.zero_grad(set_to_none=True)
            g_loss.backward()
            optG.step()
            g_losses.append(g_loss.item())
            d_losses.append(d_loss.item())

        row = {'epoch': epoch+1, 'g_loss': float(np.mean(g_losses)), 'd_loss': float(np.mean(d_losses))}
        history.append(row)
        print(row)
        if (epoch + 1) % max(1, EPOCHS // 5) == 0 or epoch == 0:
            save_sample_grid(G(fixed_noise, fixed_labels), OUTPUT_DIR / 'samples' / f'acgan_epoch_{epoch+1}.png')

    ckpt_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({'G': G.state_dict(), 'D': D.state_dict(), 'classes': CLASSES}, ckpt_path)
    pd.DataFrame(history).to_csv(OUTPUT_DIR / 'acgan_history.csv', index=False)
    print('Saved:', ckpt_path)
    return ckpt_path

ACGAN_CKPT = train_acgan()


In [ ]:

# =========================
# 6. Generate synthetic datasets
# =========================
def save_pil_tensor(img_tensor, path):
    img = denorm(img_tensor).detach().cpu()
    pil = transforms.ToPILImage()(img)
    path.parent.mkdir(parents=True, exist_ok=True)
    pil.save(path)

def generate_dcgan_dataset():
    for cls in CLASSES:
        ckpt = torch.load(DCGAN_CKPTS[cls], map_location=DEVICE)
        G = DCGANGenerator(NZ).to(DEVICE)
        G.load_state_dict(ckpt['G'])
        G.eval()
        out_dir = SYNTHETIC_DCGAN_OUT / cls / 'images'
        out_dir.mkdir(parents=True, exist_ok=True)
        existing = len(list_images(out_dir))
        if existing >= N_SYNTH_PER_CLASS and not FORCE_RETRAIN:
            print(f'DCGAN synthetic exists for {cls}: {existing}')
            continue
        for i in tqdm(range(N_SYNTH_PER_CLASS), desc=f'Generate DCGAN {cls}'):
            z = torch.randn(1, NZ, 1, 1, device=DEVICE)
            with torch.no_grad():
                img = G(z)[0]
            save_pil_tensor(img, out_dir / f'dcgan_{cls}_{i:05d}.png')

def generate_acgan_dataset():
    ckpt = torch.load(ACGAN_CKPT, map_location=DEVICE)
    G = ACGANGenerator(NZ, len(CLASSES)).to(DEVICE)
    G.load_state_dict(ckpt['G'])
    G.eval()
    for label, cls in enumerate(CLASSES):
        out_dir = SYNTHETIC_ACGAN_OUT / cls / 'images'
        out_dir.mkdir(parents=True, exist_ok=True)
        existing = len(list_images(out_dir))
        if existing >= N_SYNTH_PER_CLASS and not FORCE_RETRAIN:
            print(f'ACGAN synthetic exists for {cls}: {existing}')
            continue
        lab = torch.tensor([label], device=DEVICE)
        for i in tqdm(range(N_SYNTH_PER_CLASS), desc=f'Generate ACGAN {cls}'):
            z = torch.randn(1, NZ, 1, 1, device=DEVICE)
            with torch.no_grad():
                img = G(z, lab)[0]
            save_pil_tensor(img, out_dir / f'acgan_{cls}_{i:05d}.png')

generate_dcgan_dataset()
generate_acgan_dataset()
print('DCGAN synthetic root:', SYNTHETIC_DCGAN_OUT, 'images:', len(list_images(SYNTHETIC_DCGAN_OUT)))
print('ACGAN synthetic root:', SYNTHETIC_ACGAN_OUT, 'images:', len(list_images(SYNTHETIC_ACGAN_OUT)))


In [ ]:

# =========================
# 7. Compare DCGAN vs ACGAN using IS and FID
# =========================
class ImageOnlyDataset(Dataset):
    def __init__(self, root, transform, max_images=None):
        self.files = sorted(list_images(root))
        if max_images is not None:
            self.files = self.files[:max_images]
        self.transform = transform
    def __len__(self):
        return len(self.files)
    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert('RGB')
        return self.transform(img)

@torch.no_grad()
def inception_features(root, max_images=1000):
    weights = Inception_V3_Weights.DEFAULT
    tf = weights.transforms()
    ds = ImageOnlyDataset(root, tf, max_images=max_images)
    loader = DataLoader(ds, batch_size=32, shuffle=False, num_workers=0)
    model = inception_v3(weights=weights)
    model.fc = nn.Identity()
    model.eval().to(DEVICE)
    feats = []
    for x in tqdm(loader, desc=f'features {Path(root).name}'):
        x = x.to(DEVICE)
        f = model(x)
        feats.append(f.detach().cpu().numpy())
    return np.concatenate(feats, axis=0)

def fid_from_features(real_feats, fake_feats):
    mu1, sigma1 = real_feats.mean(axis=0), np.cov(real_feats, rowvar=False)
    mu2, sigma2 = fake_feats.mean(axis=0), np.cov(fake_feats, rowvar=False)
    diff = mu1 - mu2
    covmean = sqrtm(sigma1.dot(sigma2))
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    return float(diff.dot(diff) + np.trace(sigma1 + sigma2 - 2 * covmean))

@torch.no_grad()
def inception_score(root, max_images=1000, splits=10):
    weights = Inception_V3_Weights.DEFAULT
    tf = weights.transforms()
    ds = ImageOnlyDataset(root, tf, max_images=max_images)
    loader = DataLoader(ds, batch_size=32, shuffle=False, num_workers=0)
    model = inception_v3(weights=weights)
    model.eval().to(DEVICE)
    probs = []
    for x in tqdm(loader, desc=f'IS {Path(root).name}'):
        x = x.to(DEVICE)
        p = F.softmax(model(x), dim=1)
        probs.append(p.detach().cpu().numpy())
    probs = np.concatenate(probs, axis=0)
    scores = []
    for part in np.array_split(probs, min(splits, len(probs))):
        py = part.mean(axis=0, keepdims=True)
        kl = part * (np.log(part + 1e-10) - np.log(py + 1e-10))
        scores.append(np.exp(kl.sum(axis=1).mean()))
    return float(np.mean(scores)), float(np.std(scores))

if COMPUTE_IS_FID:
    real_feats = inception_features(LABELLED_DATA, max_images=METRIC_MAX_IMAGES)
    rows = []
    for name, root in [('ACGAN', SYNTHETIC_ACGAN_OUT), ('DCGAN', SYNTHETIC_DCGAN_OUT)]:
        fake_feats = inception_features(root, max_images=METRIC_MAX_IMAGES)
        fid = fid_from_features(real_feats, fake_feats)
        is_mean, is_std = inception_score(root, max_images=METRIC_MAX_IMAGES)
        rows.append({'model': name, 'IS_mean': is_mean, 'IS_std': is_std, 'FID': fid, 'n_images': len(list_images(root))})
    results = pd.DataFrame(rows).sort_values('FID')
    results.to_csv(OUTPUT_DIR / 'gan_comparison_metrics.csv', index=False)
    display(results)
else:
    print('COMPUTE_IS_FID=False; skipping metrics.')

print('DONE. Use this DCGAN path for COVID-QU-Syn classification runs:')
print(SYNTHETIC_DCGAN_OUT)
